# Fase 1 & 2: Análisis Exploratorio de Datos (EDA) y Perfilado Clínico

> **Proyecto:** Modelado predictivo y análisis multifactorial de riesgo de enfermedades cardíacas  
> **Institución:** Universidad Autónoma de San Luis Potosí (UASLP) — Facultad de Ingeniería  
> **Dataset:** 100,000 registros sintéticos calibrados a parámetros poblacionales (CDC / AHA / NHANES)  
> **Objetivo:** Comprender a fondo las distribuciones, calidad de datos y correlaciones diferenciales entre factores clínicos, demográficos, de estilo de vida y señales de wearables.

---

## Estructura de este Notebook Guiado
1. **Configuración y Carga de Datos** *(Pre-configurado: Trabajo de fontanería listo)*
2. **Inspección de Integridad y Calidad**
3. **Análisis de la Variable Objetivo (`heart_disease`)**
4. **Perfilado Demográfico y Socioeconómico**
5. **Marcadores Clínicos y Diagnósticos**
6. **Factores Modificables de Estilo de Vida**
7. **Biomarcadores de Dispositivos Vestibles (`hrv_score`, `avg_resting_hr`)**
8. **Ejercicios Guiados para el Alumno (TODOs)**


In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    RAW_DATA_PATH,
    TARGET_COL,
    DEMOGRAPHIC_COLS,
    LIFESTYLE_COLS,
    MEDICAL_HISTORY_COLS,
    CLINICAL_MARKER_COLS,
    WEARABLE_COLS,
    PALETTE,
    POS_COLOR,
    NEG_COLOR,
    PLOT_STYLE,
)
from src.data_loader import load_processed_splits

plt.rcParams.update(PLOT_STYLE)
print("Entorno inicializado correctamente con matplotlib, seaborn y src.config!")


## 1. Carga del Dataset y Verificación de Integridad
Utilizamos la utilidad `load_processed_splits()` de `src.data_loader` para trabajar con las particiones estratificadas ya generadas sin fuga de datos.


In [ ]:
train_df, val_df, test_df = load_processed_splits()
print(f"Conjunto de Entrenamiento: {train_df.shape}")
print(f"Conjunto de Validación:    {val_df.shape}")
print(f"Conjunto de Prueba:        {test_df.shape}")
train_df.head()


## 2. Visión General de Tipos y Distribución del Target
La prevalencia de la variable objetivo `heart_disease` es de **46.1%**, lo que proporciona un balance natural ideal sin necesidad de sobremuestreo artificial.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

target_counts = train_df[TARGET_COL].value_counts()
labels = ['Sin Enfermedad (0)', 'Enfermedad Cardíaca (1)']
bars = axes[0].bar(labels, target_counts.values, color=[NEG_COLOR, POS_COLOR], width=0.5)

for bar, val in zip(bars, target_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f"{val:,} ({val/len(train_df):.1%})", ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Distribución de Clases en Train', fontweight='bold')
axes[0].set_ylabel('Pacientes')

summary = train_df[CLINICAL_MARKER_COLS[:4] + WEARABLE_COLS].describe().T[['mean', 'std', 'min', '50%', 'max']]
axes[1].axis('off')
table = axes[1].table(cellText=summary.round(2).values, rowLabels=summary.index,
                      colLabels=['Media', 'Desv. Est.', 'Mín', 'Mediana', 'Máx'], loc='center')
table.scale(1, 1.6)
axes[1].set_title('Resumen Estadístico Seleccionado', fontweight='bold')

plt.tight_layout()
plt.show()


---
## 3. Ejercicios Guiados para el Alumno (TODOs)

A continuación, implementa los análisis clínicos y de estilo de vida esenciales para responder la pregunta de investigación:
> *¿Cuál es el factor de riesgo de sufrir una enfermedad cardíaca, evaluando el impacto diferenciado entre factores clínicos y factores de estilo de vida?*


### 🛠️ Ejercicio 1.1: Cálculo y Visualización de Riesgo Relativo / Tasa de Prevalencia
Calcula la tasa de prevalencia cardíaca (`mean()`) condicionada a cada categoría de los factores de riesgo clave:
- `hypertension` (0 vs 1)
- `diabetes` (0 vs 1)
- `smoking_status` (Never, Former, Current)
- `exercise_angina` (0 vs 1)

**Objetivo:** Determinar qué factor incrementa en mayor proporción la probabilidad empírica de evento cardíaco.


In [ ]:
# TODO: Calcula la tasa de prevalencia de heart_disease para cada grupo.
# 1. Hipertensión:
prev_hyp = train_df.groupby('hypertension')['heart_disease'].agg(['count', 'mean']).rename(columns={'count': 'Pacientes', 'mean': 'Prevalencia'})
print("=== Prevalencia por Hipertensión ===")
display(prev_hyp)

# 2. Diabetes:
prev_diab = train_df.groupby('diabetes')['heart_disease'].agg(['count', 'mean']).rename(columns={'count': 'Pacientes', 'mean': 'Prevalencia'})
print("=== Prevalencia por Diabetes ===")
display(prev_diab)

# 3. Tabaquismo (smoking_status):
prev_smoke = train_df.groupby('smoking_status')['heart_disease'].agg(['count', 'mean']).rename(columns={'count': 'Pacientes', 'mean': 'Prevalencia'})
print("=== Prevalencia por Tabaquismo ===")
display(prev_smoke)

# 4. Angina inducida por ejercicio (exercise_angina):
prev_angina = train_df.groupby('exercise_angina')['heart_disease'].agg(['count', 'mean']).rename(columns={'count': 'Pacientes', 'mean': 'Prevalencia'})
print("=== Prevalencia por Angina de Ejercicio ===")
display(prev_angina)


### 🛠️ Ejercicio 1.2: Análisis de la No-Linealidad del Sueño (U-Shaped Curve)
La literatura médica sugiere que tanto la privación de sueño (<6 horas) como el sueño excesivo (>9 horas) se asocian a un mayor riesgo cardiovascular.

**Tu tarea:**
1. Agrupa la columna `sleep_hours` en rangos o redondea a horas enteras con `train_df['sleep_hours'].round()`.
2. Calcula la tasa promedio de `heart_disease` por cada intervalo.
3. Grafica la curva resultante (línea o barras) y confirma visualmente si se observa el patrón en forma de 'U'.


In [ ]:
# TODO: Implementa la agrupación y gráfica de sleep_hours vs heart_disease
sleep_rounded = train_df['sleep_hours'].round()
sleep_grouped = train_df.groupby(sleep_rounded)['heart_disease'].agg(['count', 'mean']).reset_index()

plt.figure(figsize=(9, 4.5))
plt.plot(sleep_grouped['sleep_hours'], sleep_grouped['mean'] * 100, marker='o', linewidth=2.5, color='#eb6834')
plt.axvline(7.5, color='#1baf7a', linestyle='--', label='Duración Óptima (~7.5 hrs)')
plt.title('Tasa de Enfermedad Cardíaca vs Horas de Sueño (Curva en U)', fontweight='bold')
plt.xlabel('Horas de Sueño (redondeadas)')
plt.ylabel('Prevalencia de Enfermedad Cardíaca (%)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


### 🛠️ Ejercicio 1.3: Análisis de Wearables (HRV Score y Frecuencia Cardíaca)
El biomarcador `hrv_score` (Variabilidad de la Frecuencia Cardíaca) refleja la actividad parasimpática y la salud del sistema nervioso autónomo.
Una menor HRV se asocia clínicamente con fatiga cardíaca y mayor mortalidad.

**Tu tarea:**
1. Grafica la distribución de `hrv_score` y `avg_resting_hr` comparando pacientes sanos (`heart_disease=0`) vs con enfermedad (`heart_disease=1`).
2. Compara la media y desviación estándar de ambos grupos.


In [ ]:
# TODO: Genera histogramas superpuestos o diagramas boxplot para hrv_score y avg_resting_hr
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# HRV Score por condición cardíaca
sns.kdeplot(data=train_df[train_df['heart_disease'] == 0]['hrv_score'], ax=axes[0], fill=True, label='Sano (0)', color='#2a78d6')
sns.kdeplot(data=train_df[train_df['heart_disease'] == 1]['hrv_score'], ax=axes[0], fill=True, label='Enfermedad (1)', color='#e34948')
axes[0].set_title('Distribución de HRV Score según Condición Cardíaca', fontweight='bold')
axes[0].set_xlabel('HRV Score (ms)')
axes[0].legend()

# Frecuencia Cardíaca en Reposo
sns.boxplot(x='heart_disease', y='avg_resting_hr', data=train_df, ax=axes[1], palette=['#2a78d6', '#e34948'])
axes[1].set_title('Frecuencia Cardíaca en Reposo por Grupo', fontweight='bold')
axes[1].set_xticklabels(['Sano (0)', 'Enfermedad (1)'])
axes[1].set_ylabel('Frecuencia Cardíaca Media en Reposo (lpm)')

plt.tight_layout()
plt.show()

# Estadísticos comparativos
stats_wearables = train_df.groupby('heart_disease')[['hrv_score', 'avg_resting_hr']].agg(['mean', 'std']).T
display(stats_wearables)


### 🛠️ Ejercicio 1.4: Síntesis de Correlaciones (Clínicos vs Estilo de Vida)
Calcula las correlaciones de Pearson y Spearman de las variables con respecto a `heart_disease`.

**Preguntas de reflexión para tu reporte:**
1. ¿Cuáles son las 3 variables con mayor correlación con la enfermedad? ¿Pertenecen al grupo clínico o al grupo de estilo de vida?
2. ¿Existe alguna variable de estilo de vida con correlación comparable a los marcadores clínicos?


In [ ]:
# TODO: Codifica o selecciona variables numéricas y calcula la correlación con TARGET_COL.
corr_df = train_df.copy()
for col in corr_df.select_dtypes(include=['object', 'category']).columns:
    corr_df[col] = pd.factorize(corr_df[col])[0]

correlations = corr_df.corr(method='pearson')['heart_disease'].drop('heart_disease').sort_values(ascending=True)

plt.figure(figsize=(10, 8))
colors = ['#e34948' if c > 0.20 else '#2a78d6' if c < -0.20 else '#7f8c8d' for c in correlations.values]
correlations.plot(kind='barh', color=colors)
plt.title('Correlación de Pearson con Enfermedad Cardíaca (TARGET)', fontweight='bold')
plt.xlabel('Coeficiente de Correlación (r)')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

print("Top 5 factores de mayor correlación absoluta:")
display(correlations.abs().sort_values(ascending=False).head(5))
